IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("🔗 PRODUCT FATIGUE INTEGRATION ANALYSIS")
print("="*70)

1. LOAD ALL FATIGUE SIGNALS

In [ ]:
print("\n📦 Loading processed fatigue signals from all datasets...\n")

try:
    reviews = pd.read_csv('../data/processed/reviews_fatigue_signals.csv')
    print(f"✅ Reviews loaded: {reviews.shape}")
    # Convert month to period if string
    if reviews['month'].dtype == 'object':
        reviews['month'] = pd.to_datetime(reviews['month']).dt.to_period('M')
except FileNotFoundError:
    print("❌ Reviews file not found. Please run 01_reviews_eda.ipynb first.")
    reviews = None
except Exception as e:
    print(f"❌ Error loading reviews: {e}")
    reviews = None

try:
    sales = pd.read_csv('../data/processed/sales_fatigue_signals.csv')
    print(f"✅ Sales loaded: {sales.shape}")
    # Convert month to period if string
    if sales['month'].dtype == 'object':
        sales['month'] = pd.to_datetime(sales['month']).dt.to_period('M')
except FileNotFoundError:
    print("❌ Sales file not found. Please run 02_sales_eda.ipynb first.")
    sales = None
except Exception as e:
    print(f"❌ Error loading sales: {e}")
    sales = None

try:
    usage = pd.read_csv('../data/processed/usage_fatigue_signals.csv')
    print(f"✅ Usage loaded: {usage.shape}")
    # Convert month to period if string
    if usage['month'].dtype == 'object':
        usage['month'] = pd.to_datetime(usage['month']).dt.to_period('M')
except FileNotFoundError:
    print("❌ Usage file not found. Please run 03_usage_eda.ipynb first.")
    usage = None
except Exception as e:
    print(f"❌ Error loading usage: {e}")
    usage = None

# Check if at least one dataset loaded
available_datasets = sum([reviews is not None, sales is not None, usage is not None])
if available_datasets == 0:
    raise ValueError("❌ No datasets loaded. Please run EDA notebooks first.")
elif available_datasets < 3:
    print(f"\n⚠️ Warning: Only {available_datasets}/3 datasets loaded.")
    print("   Integration analysis will be limited.\n")

2. DATASET OVERVIEW

In [ ]:
print("\n" + "="*70)
print("📊 DATASET OVERVIEW")
print("="*70)

datasets_info = []

if reviews is not None:
    datasets_info.append({
        'Dataset': 'Reviews (Emotional)',
        'Rows': f"{len(reviews):,}",
        'Columns': len(reviews.columns),
        'Products': f"{reviews['ProductId'].nunique():,}",
        'Time Period': f"{reviews['month'].min()} to {reviews['month'].max()}"
    })

if sales is not None:
    datasets_info.append({
        'Dataset': 'Sales (Financial)',
        'Rows': f"{len(sales):,}",
        'Columns': len(sales.columns),
        'Products': f"{sales['StockCode'].nunique():,}",
        'Time Period': f"{sales['month'].min()} to {sales['month'].max()}"
    })

if usage is not None:
    datasets_info.append({
        'Dataset': 'Usage (Behavioral)',
        'Rows': f"{len(usage):,}",
        'Columns': len(usage.columns),
        'Products': f"{usage['product_id'].nunique():,}",
        'Time Period': f"{usage['month'].min()} to {usage['month'].max()}"
    })

if datasets_info:
    overview_df = pd.DataFrame(datasets_info)
    print("\n" + overview_df.to_string(index=False))


3. FATIGUE DISTRIBUTION SUMMARY

In [ ]:
print("\n" + "="*70)
print("🎯 FATIGUE DISTRIBUTION ACROSS DATASETS")
print("="*70)

fatigue_summary = {}

if reviews is not None:
    print("\n1️⃣ EMOTIONAL FATIGUE (Reviews):")
    reviews_dist = reviews['fatigue_label'].value_counts()
    print(reviews_dist)
    print(f"\n   Total fatigue rate: {(reviews_dist.get('high_fatigue', 0) + reviews_dist.get('moderate_fatigue', 0)) / len(reviews) * 100:.2f}%")
    fatigue_summary['Reviews'] = reviews_dist

if sales is not None:
    print("\n2️⃣ FINANCIAL FATIGUE (Sales):")
    sales_dist = sales['fatigue_label'].value_counts()
    print(sales_dist)
    print(f"\n   Total fatigue rate: {(sales_dist.get('high_fatigue', 0) + sales_dist.get('moderate_fatigue', 0)) / len(sales) * 100:.2f}%")
    fatigue_summary['Sales'] = sales_dist

if usage is not None:
    print("\n3️⃣ BEHAVIORAL FATIGUE (Usage):")
    usage_dist = usage['fatigue_label'].value_counts()
    print(usage_dist)
    print(f"\n   Total fatigue rate: {(usage_dist.get('high_fatigue', 0) + usage_dist.get('moderate_fatigue', 0)) / len(usage) * 100:.2f}%")
    fatigue_summary['Usage'] = usage_dist


4. VISUALIZATION: COMPARATIVE FATIGUE RATES

In [ ]:
print("\n" + "="*70)
print("📊 GENERATING COMPARATIVE VISUALIZATIONS")
print("="*70)

# Build fatigue rates dataframe dynamically
fatigue_data = []

if reviews is not None:
    fatigue_data.append({
        'Dataset': 'Reviews',
        'High Fatigue': (reviews['fatigue_label'] == 'high_fatigue').mean() * 100,
        'Moderate Fatigue': (reviews['fatigue_label'] == 'moderate_fatigue').mean() * 100,
        'Healthy': (reviews['fatigue_label'] == 'healthy').mean() * 100
    })

if sales is not None:
    fatigue_data.append({
        'Dataset': 'Sales',
        'High Fatigue': (sales['fatigue_label'] == 'high_fatigue').mean() * 100,
        'Moderate Fatigue': (sales['fatigue_label'] == 'moderate_fatigue').mean() * 100,
        'Healthy': (sales['fatigue_label'] == 'healthy').mean() * 100
    })

if usage is not None:
    fatigue_data.append({
        'Dataset': 'Usage',
        'High Fatigue': (usage['fatigue_label'] == 'high_fatigue').mean() * 100,
        'Moderate Fatigue': (usage['fatigue_label'] == 'moderate_fatigue').mean() * 100,
        'Healthy': (usage['fatigue_label'] == 'healthy').mean() * 100
    })

fatigue_rates = pd.DataFrame(fatigue_data)

# Visualization 1: Comparative Bar Chart
if len(fatigue_rates) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(fatigue_rates))
    width = 0.25
    
    bars1 = ax.bar(x - width, fatigue_rates['High Fatigue'], width, 
                   label='High Fatigue', color='#e74c3c', alpha=0.9)
    bars2 = ax.bar(x, fatigue_rates['Moderate Fatigue'], width, 
                   label='Moderate Fatigue', color='#f39c12', alpha=0.9)
    bars3 = ax.bar(x + width, fatigue_rates['Healthy'], width, 
                   label='Healthy', color='#27ae60', alpha=0.9)
    
    ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_title('Product Fatigue Distribution Across Dimensions', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(fatigue_rates['Dataset'], fontsize=11)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_ylim(0, 100)
    
    # Add value labels
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:  # Only show label if value > 0
                ax.text(bar.get_x() + bar.get_width()/2., height,
                        f'{height:.1f}%',
                        ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../outputs/comparative_fatigue_rates.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("   ✅ Saved: outputs/comparative_fatigue_rates.png")

5. CROSS-DATASET INSIGHTS

In [ ]:
print("\n" + "="*70)
print("🔍 CROSS-DATASET INSIGHTS")
print("="*70)

# Visualization 2: Stacked Bar Chart
if len(fatigue_rates) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    fatigue_rates_stacked = fatigue_rates.set_index('Dataset')[
        ['Healthy', 'Moderate Fatigue', 'High Fatigue']
    ]
    
    fatigue_rates_stacked.plot(
        kind='barh',
        stacked=True,
        ax=ax,
        color=['#27ae60', '#f39c12', '#e74c3c'],
        width=0.7
    )
    
    ax.set_xlabel('Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
    ax.set_title('Stacked Fatigue Distribution by Dataset', 
                 fontsize=14, fontweight='bold', pad=20)
    ax.legend(title='Fatigue Level', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_xlim(0, 100)
    
    plt.tight_layout()
    plt.savefig('../outputs/stacked_fatigue_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("   ✅ Saved: outputs/stacked_fatigue_distribution.png")

6. TEMPORAL ALIGNMENT ANALYSIS

In [ ]:
print("\n" + "="*70)
print("📅 TEMPORAL COVERAGE ANALYSIS")
print("="*70)

temporal_data = []

if reviews is not None:
    temporal_data.append({
        'Dataset': 'Reviews',
        'Start Date': str(reviews['month'].min()),
        'End Date': str(reviews['month'].max()),
        'Duration (Months)': reviews['month'].nunique(),
        'Avg Products/Month': f"{len(reviews) / reviews['month'].nunique():.0f}"
    })

if sales is not None:
    temporal_data.append({
        'Dataset': 'Sales',
        'Start Date': str(sales['month'].min()),
        'End Date': str(sales['month'].max()),
        'Duration (Months)': sales['month'].nunique(),
        'Avg Products/Month': f"{len(sales) / sales['month'].nunique():.0f}"
    })

if usage is not None:
    temporal_data.append({
        'Dataset': 'Usage',
        'Start Date': str(usage['month'].min()),
        'End Date': str(usage['month'].max()),
        'Duration (Months)': usage['month'].nunique(),
        'Avg Products/Month': f"{len(usage) / usage['month'].nunique():.0f}"
    })

if temporal_data:
    temporal_df = pd.DataFrame(temporal_data)
    print("\n" + temporal_df.to_string(index=False))

 7. KEY FATIGUE METRICS COMPARISON

In [ ]:
print("\n" + "="*70)
print("📈 KEY METRICS SUMMARY")
print("="*70)

metrics_summary = []

if reviews is not None:
    metrics_summary.append({
        'Dataset': 'Reviews',
        'Dimension': 'Emotional',
        'Primary Signal': 'Sentiment Velocity',
        'Avg Signal Value': f"{reviews['sentiment_velocity'].mean():.2f}%",
        'High Fatigue Products': reviews[reviews['fatigue_label'] == 'high_fatigue']['ProductId'].nunique()
    })

if sales is not None:
    metrics_summary.append({
        'Dataset': 'Sales',
        'Dimension': 'Financial',
        'Primary Signal': 'Revenue Velocity',
        'Avg Signal Value': f"{sales['revenue_velocity'].mean():.2f}%",
        'High Fatigue Products': sales[sales['fatigue_label'] == 'high_fatigue']['StockCode'].nunique()
    })

if usage is not None:
    metrics_summary.append({
        'Dataset': 'Usage',
        'Dimension': 'Behavioral',
        'Primary Signal': 'Engagement Velocity',
        'Avg Signal Value': f"{usage['engagement_velocity'].mean():.2f}%",
        'High Fatigue Products': usage[usage['fatigue_label'] == 'high_fatigue']['product_id'].nunique()
    })

if metrics_summary:
    metrics_df = pd.DataFrame(metrics_summary)
    print("\n" + metrics_df.to_string(index=False))

8. EXPORT INTEGRATION SUMMARY

In [ ]:
print("\n" + "="*70)
print("💾 EXPORTING INTEGRATION SUMMARY")
print("="*70)

# Save fatigue rates summary
if len(fatigue_rates) > 0:
    fatigue_rates.to_csv('../data/processed/fatigue_rates_comparison.csv', index=False)
    print("   ✅ Saved: data/processed/fatigue_rates_comparison.csv")

# Save temporal summary
if temporal_data:
    temporal_df.to_csv('../data/processed/temporal_coverage_summary.csv', index=False)
    print("   ✅ Saved: data/processed/temporal_coverage_summary.csv")

# Save metrics summary
if metrics_summary:
    metrics_df.to_csv('../data/processed/key_metrics_summary.csv', index=False)
    print("   ✅ Saved: data/processed/key_metrics_summary.csv")

9. INTEGRATION READINESS CHECK

In [ ]:
print("\n" + "="*70)
print("✅ INTEGRATION ANALYSIS COMPLETE!")
print("="*70)

print("\n🎯 NEXT STEPS FOR MODEL DEVELOPMENT:")

if available_datasets == 3:
    print("\n   ✅ All 3 datasets successfully integrated")
    print("   ✅ Multi-dimensional fatigue signals ready")
    print("   ✅ Ready for unified modeling approach")
    print("\n   📋 Recommended modeling strategy:")
    print("      1. Feature alignment across datasets")
    print("      2. Ensemble model combining all dimensions")
    print("      3. Cross-validation on temporal splits")
elif available_datasets == 2:
    print("\n   ⚠️ 2/3 datasets available")
    print("   ⚠️ Consider dual-dimension model")
    print("   ⚠️ Or complete missing dataset first")
elif available_datasets == 1:
    print("\n   ⚠️ Only 1 dataset available")
    print("   ⚠️ Single-dimension model possible")
    print("   ⚠️ But multi-dimensional approach recommended")

print("\n" + "="*70)